[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Rate Limits &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up, including the practice API, `wait_for_reset`,
`get_patiently`, `retry_after_seconds`, `backoff` and `PoliteClient`. Run it first. The limit runs on
real time, so every answer that spends it starts with `wait_for_reset`, which makes it print the same
thing however soon it follows the one before.


In [1]:
import email.utils
import importlib
import random
import sys
import time
import urllib.request
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()


def wait_for_reset():
    """Wait until the limit's window closes, if any of its requests have been spent."""
    limit = requests.get(f"{BASE}/rate-limit", timeout=10).json()
    if limit["remaining"] < limit["limit"]:
        time.sleep(limit["reset"])


def get_patiently(url, attempts=3, **kwargs):
    """A response, sent again after each 429 once Retry-After's seconds have passed, at most `attempts` times."""
    for attempt in range(1, attempts + 1):
        response = requests.get(url, timeout=10, **kwargs)
        if response.status_code != 429 or attempt == attempts:
            response.raise_for_status()
            return response
        wait = int(response.headers["Retry-After"])
        print(f"  429 on attempt {attempt}: waiting {wait} seconds")
        time.sleep(wait)


def retry_after_seconds(response):
    """The seconds a 429 asks a client to wait, from a Retry-After of seconds or of a date."""
    value = response.headers["Retry-After"]
    if value.isdigit():
        return int(value)
    retry_at = email.utils.parsedate_to_datetime(value)
    sent_at = email.utils.parsedate_to_datetime(response.headers["Date"])
    return max(0, (retry_at - sent_at).total_seconds())


def backoff(attempt, rng, cap=30):
    """Seconds to wait after refusal number attempt + 1: a random share of a longest that doubles each time."""
    return rng.uniform(0, min(cap, 2 ** attempt))


class PoliteClient:
    """Requests to one API that stay inside its rate limit, and give up rather than wait too long."""

    def __init__(self, base, user_agent, max_attempts=4, max_wait=10, seed=None):
        self.base = base
        self.headers = {"User-Agent": user_agent}
        self.max_attempts = max_attempts
        self.max_wait = max_wait
        self.rng = random.Random(seed)
        self.resume_at = 0.0                   # a time.monotonic() reading: no request goes before it
        self.refused = 0
        self.pauses = 0

    def wait_until_allowed(self, path):
        seconds = self.resume_at - time.monotonic()
        if seconds > self.max_wait:
            raise RuntimeError(f"{path} asked for a wait of {seconds:.0f} seconds, more than {self.max_wait}")
        if seconds > 0:
            time.sleep(seconds)
            self.pauses += 1

    def get(self, path, **params):
        """The JSON at path, sent no sooner than the limit allows."""
        for attempt in range(1, self.max_attempts + 1):
            self.wait_until_allowed(path)
            response = requests.get(f"{self.base}{path}", params=params, headers=self.headers, timeout=10)
            if response.status_code == 429 and attempt < self.max_attempts:
                self.refused += 1
                if "Retry-After" in response.headers:
                    wait = retry_after_seconds(response)
                else:
                    wait = backoff(attempt - 1, self.rng, cap=self.max_wait)
                self.resume_at = time.monotonic() + wait
                continue
            response.raise_for_status()
            if response.headers.get("X-RateLimit-Remaining") == "0":
                self.resume_at = time.monotonic() + int(response.headers["X-RateLimit-Reset"])
            return response.json()


print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** One request, and what it says about the limit.


In [2]:
wait_for_reset()
response = requests.get(f"{BASE}/network/latest", params={"station": "tromso"}, timeout=10)

print(response.json())
for name in ["X-RateLimit-Limit", "X-RateLimit-Remaining", "X-RateLimit-Reset"]:
    print(f"{name}: {response.headers[name]}")


{'station': 'tromso', 'time': '2026-03-01T09:00Z', 'temperature_c': -6.3}
X-RateLimit-Limit: 5
X-RateLimit-Remaining: 4
X-RateLimit-Reset: 2


Tromso's latest reading, and one of the five requests spent. The window opened with this request,
and closes 2 seconds after it.


**2.** The limit before and after three requests.


In [3]:
wait_for_reset()
print("before:", requests.get(f"{BASE}/rate-limit", timeout=10).json())
for _ in range(3):
    requests.get(f"{BASE}/network/latest", timeout=10)
print("after: ", requests.get(f"{BASE}/rate-limit", timeout=10).json())


before: {'limit': 5, 'remaining': 5, 'reset': 0}
after:  {'limit': 5, 'remaining': 2, 'reset': 2}


Three requests spent three of the five, and the two requests to `/rate-limit` spent nothing. Before
the first of the three, no window was open, so `reset` was 0.


**3.** Requests until the first refusal.


In [4]:
wait_for_reset()
answered = 0
while True:
    response = requests.get(f"{BASE}/network/latest", timeout=10)
    if response.status_code == 429:
        break
    answered += 1

print(answered, "answered | Retry-After:", response.headers["Retry-After"])


5 answered | Retry-After: 2


The sixth request was the first refused, and its `Retry-After` is the time left in the window. This
loop ends only because the endpoint has a limit. Against one without a limit it would never stop, so
a loop like it in a real program also needs a most.


**4.** Seven readings, with a wait in the middle.


In [5]:
wait_for_reset()
temperatures = [get_patiently(f"{BASE}/network/latest", params={"station": "oslo"}).json()["temperature_c"]
                for _ in range(7)]

print(temperatures)


  429 on attempt 1: waiting 2 seconds
[-4.2, -4.2, -4.2, -4.2, -4.2, -4.2, -4.2]


The sixth request met the limit, waited, and was answered in a new window, which the seventh used
too. The latest reading did not change in those two seconds, a reminder that the cheapest request is
the one not sent: one reading, kept, would have served all seven.


**5.** Five waits from a seeded generator.


In [6]:
rng = random.Random(7)             # seeded, so that the waits are the same on every run
waits = [round(backoff(attempt, rng), 2) for attempt in range(5)]

print(waits, "| total:", round(sum(waits), 2))


[0.32, 0.3, 2.6, 0.58, 8.57] | total: 12.37


Each wait falls below its longest, 1, 2, 4, 8 and 16 seconds, and any of them can be short. A
different seed gives different waits, and a real client uses no seed at all.


**6.** A client that will not wait long.


In [7]:
wait_for_reset()
client = PoliteClient(BASE, "station-report/1.0 (reports@example.com)", max_wait=1)
readings = []
try:
    for _ in range(6):
        readings.append(client.get("/network/latest", station="bergen"))
except RuntimeError as error:
    print(len(readings), "readings arrived, then:", error)


5 readings arrived, then: /network/latest asked for a wait of 2 seconds, more than 1


Five readings arrived. The fifth response left no requests in the window, so the client set its next
request 2 seconds away, and with `max_wait` at 1 it raised an error instead of waiting. The fifth
reading was not lost, because the client waits before a request, not after a response. A program
that must never stall uses a small `max_wait`, and handles the error, perhaps by keeping the readings
it has.


---

&#8592; **Back to:** [Rate Limits](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/11-rate-limits.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
